In [1]:
%load_ext autoreload
%autoreload 2

import sys, os
_root = os.path.abspath(os.getcwd())
while not os.path.exists(os.path.join(_root, "config.py")) and os.path.dirname(_root) != _root:
    _root = os.path.dirname(_root)
if _root not in sys.path: sys.path.insert(0, _root)
import numpy as np, pandas as pd
import config
from cprics.io_data import load_peak_amplitudes
from cprics.continuous import build_all_combined, CANDIDATE_FLOOR
from cprics.tracking import track_islands

BANDS04 = {"4.5": "45_top_right", "6": "6_top_right", "9": "9_top_right",
           "13.5": "135_top_right", "18": "18_top_right"}
WLMAP   = {"4.5": 4.5, "6": 6.0, "9": 9.0, "13.5": 13.5, "18": 18.0}

In [2]:
rows, band_sites = [], {}
for name, chan in BANDS04.items():
    islb = track_islands(build_all_combined(load_peak_amplitudes(config.PEAK_AMP_DIR, chan),
                                            wavelength=WLMAP[name], candidate_floor=CANDIDATE_FLOOR))
    g = islb.groupby("site_id").agg(
        ELR=("ELR", "first"), TrackID=("TrackID", "first"),
        start=("start_location", "min"), end=("end_location", "max"),
        max_hard=("hard_amplitude", "max"), n_obs=("date", "nunique"))
    band_sites[name] = g      # kept for the cross-band overlap step next
    rows.append({"band": name + "m", "n_sites": len(g),
                 "p50": round(g.max_hard.quantile(.5), 1), "p90": round(g.max_hard.quantile(.9), 1),
                 "p99": round(g.max_hard.quantile(.99), 1), "max": round(g.max_hard.max(), 1),
                 "concern_18": int((g.max_hard >= 18).sum()),
                 "critical_30": int((g.max_hard >= 30).sum())})
print(pd.DataFrame(rows).to_string(index=False))

 band  n_sites  p50  p90  p99  max  concern_18  critical_30
 4.5m      776  0.0  0.0  0.0 18.6           1            0
   6m     2211  0.0  0.0  0.0 40.6           5            1
   9m     2973  0.0  0.0 30.3 80.7          99           31
13.5m     2450  0.0  0.0 23.9 64.5          53           11
  18m     2374  0.0  0.0 29.7 71.3          76           23


In [3]:
MAIN = ["9", "13.5", "18"]        # 4.5/6 negligible, dropped
crit = []
for name in MAIN:
    c = band_sites[name]; c = c[c.max_hard >= 30].reset_index()
    c["band"] = name + "m"; c["center"] = (c.start + c.end) / 2
    crit.append(c[["band", "ELR", "TrackID", "start", "end", "center", "max_hard"]])
crit = pd.concat(crit, ignore_index=True).sort_values(["ELR", "TrackID", "center"]).reset_index(drop=True)

GAP = 10.0                        # criticals whose footprints are within 10m at the same ELR/TrackID = one location
loc_id, counter, pe, pELR, pTID = [], 0, None, None, None
for _, r in crit.iterrows():
    if pe is not None and r.ELR == pELR and r.TrackID == pTID and (r.start - pe) <= GAP:
        loc_id.append(counter); pe = max(pe, r.end)
    else:
        counter += 1; loc_id.append(counter); pe, pELR, pTID = r.end, r.ELR, r.TrackID
crit["loc"] = loc_id

pivot = crit.assign(v=1).pivot_table(index="loc", columns="band", values="v", aggfunc="max", fill_value=0)
print("distinct critical locations (9/13.5/18m union):", len(pivot))
print("\nlocations critical in each band:\n" + pivot.sum().to_string())
print(f"\ncritical at 13.5m or 18m but NOT at 9m (missed by 9m-only screening): {int((pivot.get('9m', 0) == 0).sum())}")
print("\noverlap patterns (9m,13.5m,18m -> count):")
print(pivot.groupby(list(pivot.columns)).size().to_string())

distinct critical locations (9/13.5/18m union): 36

locations critical in each band:
band
13.5m     9
18m      14
9m       18

critical at 13.5m or 18m but NOT at 9m (missed by 9m-only screening): 18

overlap patterns (9m,13.5m,18m -> count):
13.5m  18m  9m
0      0    1     15
       1    0     10
            1      2
1      0    0      7
       1    0      1
            1      1


In [4]:
def summarise(GAP):
    cc = crit.sort_values(["ELR", "TrackID", "center"]).reset_index(drop=True)
    loc, k, pe, pELR, pTID = [], 0, None, None, None
    for _, r in cc.iterrows():
        if pe is not None and r.ELR == pELR and r.TrackID == pTID and (r.start - pe) <= GAP:
            loc.append(k); pe = max(pe, r.end)
        else:
            k += 1; loc.append(k); pe, pELR, pTID = r.end, r.ELR, r.TrackID
    cc["loc"] = loc
    pv = cc.assign(v=1).pivot_table(index="loc", columns="band", values="v", aggfunc="max", fill_value=0)
    return len(pv), int((pv.get("9m", 0) == 0).sum())

for GAP in [0.0, 5.0, 10.0, 20.0]:
    n, m = summarise(GAP)
    print(f"GAP={GAP:4.0f}m   distinct critical locations={n}   missed-by-9m-only={m}")

GAP=   0m   distinct critical locations=36   missed-by-9m-only=18
GAP=   5m   distinct critical locations=36   missed-by-9m-only=18
GAP=  10m   distinct critical locations=36   missed-by-9m-only=18
GAP=  20m   distinct critical locations=36   missed-by-9m-only=18


In [5]:
CONCERN, CRITICAL, MAINB = 18.0, 30.0, ["9", "13.5", "18"]

flagged = []
for name in MAINB:
    f = band_sites[name]
    f = f[f.max_hard >= CONCERN].reset_index()
    f["band"] = name + "m"; f["center"] = (f.start + f.end) / 2
    flagged.append(f[["band", "ELR", "TrackID", "start", "end", "center", "max_hard"]])
flagged = pd.concat(flagged, ignore_index=True).sort_values(["ELR", "TrackID", "center"]).reset_index(drop=True)

loc, k, pe, pELR, pTID = [], 0, None, None, None          # cluster by footprint overlap (GAP=0, proven robust)
for _, r in flagged.iterrows():
    if pe is not None and r.ELR == pELR and r.TrackID == pTID and (r.start - pe) <= 0:
        loc.append(k); pe = max(pe, r.end)
    else:
        k += 1; loc.append(k); pe, pELR, pTID = r.end, r.ELR, r.TrackID
flagged["loc"] = loc

locs = flagged.groupby("loc").agg(
    ELR=("ELR", "first"), TrackID=("TrackID", "first"),
    bands=("band", lambda s: "+".join(sorted(set(s), key=lambda b: float(b[:-1])))),
    worst_hard=("max_hard", "max"))
locs["critical"] = locs.worst_hard >= CRITICAL
has9 = set(flagged.loc[flagged.band == "9m", "loc"])

print(f"watchlist locations:  9m-only {len(has9)}  ->  multi-band {len(locs)}  (+{len(locs) - len(has9)})")
cl = locs[locs.critical]
print(f"critical locations:   9m-only {int(cl.index.isin(has9).sum())}/{len(cl)}  ->  multi-band {len(cl)}/{len(cl)}")
print("\nnew critical locations 9m misses, by band:\n" + cl[~cl.index.isin(has9)].bands.value_counts().to_string())

watchlist locations:  9m-only 57  ->  multi-band 100  (+43)
critical locations:   9m-only 24/36  ->  multi-band 36/36

new critical locations 9m misses, by band:
bands
18m          5
13.5m+18m    4
13.5m        3
